In [119]:
using JuMP, Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Clarabel, Nemo, HiGHS,COSMO,Gurobi, CDDLib
import Mosek, MosekTools

include("fun.jl")

alphamatrices (generic function with 1 method)

In [120]:
function SDP(n,m, model) # Eq. (31) from https://arxiv.org/abs/2603.19901 for self-dual codes + relaxing the m-uniform conditions, 
                         # i.e. x_ijtp = 0 when 0 < i,j,i+j-t-p =< m is not applied.

    f4, size = f4_fun(n)
    f2, size2 = f2_fun(n)

    # # variables 

    @variable(model, x[0:size-1])

    set_attribute(model, "tol_feas", 1e-10)

    A=[x[f4[i,0,0,0]]*gamma(i,0,0,0,n) for i in 0:n]

    B=[sum([Kw(i, k, n)*A[k+1]/2^n for k in 0:n]) for i in 0:n]

    S=[sum([(-1)^k*Kw(i, k, n)*A[k+1]/2^n for k in 0:n]) for i in 0:n]

    # obj funct

    Ap = [sum([comb(n-i,n-j)*A[i+1] for i in 0:j])/2^j for j in 0:n] # Relation between purities and expectation values (see https://arxiv.org/abs/quant-ph/9612015) 

    @objective(model, Min, Ap[m+1])

    # # constraints

    @constraint(model,A[0+1]==1)

    # self-dual condition

    @constraint(model, KLconst1[i=0:n], B[i+1]==A[i+1])

    # # Shadow

    @constraint(model, Shadow[i=0:n], S[i+1]>=0)

    # pure code conditions

 
    for (i, j, t, p) in keys(f4)
        if (t - p) % 2 == 1  
            @constraint(model,x[f4[i,j,t,p]]==0)
        end
    end

    arr=[j for j in 0:n  if (n−j) % 2 == 1] 
    @constraint(model, Shadowpure[i=1:length(arr)], S[arr[i]+1]==0)

    # # structure constraints

    for (i, j, t, p) in keys(f4) 
        for (a, b, c, q) in keys(f4)
            if (t - p) % 2 == 0 && t - p == c - q && [i, j, i + j - t - p] in collect(permutations((a, b, a + b - c - q)))
                
                @constraint(model,x[f4[i, j, t, p]] == x[f4[a, b, c, q]])
            
            end
        end
    end

    # # projector constraints

    @constraint(model,Pconst[k=1:n],
        sum([(2 - kron_delta(i, j))*x[f4[i,j,t,p]]*gamma(i,j,t,p,n) for (i, j, t, p) in keys(f4) if k == i + j - t - p ])
            ==(2^n)*x[f4[k, 0, 0, 0]]*gamma(k,0,0,0,n))

    # sdp constraints


    X=alphamatrices(x, f4, n)

    @constraint(model, PSDconst[i=0:size2-1], X[i+1] >=0, PSDCone())

    optimize!(model)

    return objective_value(model)/comb(n,m)
end


SDP (generic function with 2 methods)

In [122]:
n=7
m=Int(floor(n/2))

SDP(n,m, Model(Clarabel.Optimizer)) # 

-------------------------------------------------------------
           Clarabel.jl v0.11.1  -  Clever Acronym              

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 200
  constraints   = 588
  nnz(P)        = 0
  nnz(A)        = 2237
  cones (total) = 22
    : Zero        = 1,  numel = 380
    : Nonnegative = 5,  numel = (8,1,1,1,1)
    : PSDTriangle = 16,  numel = (36,21,10,3,...,3)

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-10, tol_gap_abs = 1.0e-08, tol_gap_rel = 1.0e-08,
  static reg : on, ϵ1 = 1.0e-08, ϵ2 = 4.9e-32
  dynamic reg: on, ϵ = 1.0e-13, δ = 2.0e-07
  iter refine: on, reltol = 1.0e-13, abstol = 1.0e-12, 
               max iter = 10, stop ratio = 5.0
  equilibrate: on, min_scale = 1.0e-04, max_scal

0.1278118605298215